# Train the Hangman BiGRU on Kaggle GPU

Clones the `approach/gru` branch of the project repo and runs training there.
Same masked-language-model methodology as `approach/bilstm` (which validated
at 47.7% win rate), swapping the LSTM for a GRU (fewer gates, no separate
cell state) to compare directly.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/gru"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train

Char-level BiGRU trained with a masked-language-model objective: randomly
mask letters in each training word, predict the true letter at each masked
position from bidirectional context. At inference: feed the real board
mask through the model, sum per-position letter probabilities across all
blanks, guess the highest-scoring unguessed letter.

In [ ]:
!python src/train_gru.py --epochs 20

## Validate

Same methodology as the other branches, for a fair comparison: hold out
10% of train.txt, play full interactive games against words the model
never trained on.

In [ ]:
!python src/validate_gru.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the model
still in this session. Sandbox leaderboard checkpoint only -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list.

250,000 words, one game at a time (not batched) -- prints progress every
20,000 words with an ETA. The BiLSTM branch's equivalent run took ~46
minutes end to end; expect something similar.

In [ ]:
!python src/generate_submission_gru.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/gru_masker.pt", "/kaggle/working/gru_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved gru_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")